# Giai đoạn 3 · Bước 6B — Final Test Holdout (NHANES 2021–2023)

**Bạn là người tự chạy** từng ô trong Jupyter. Notebook chưa chạy và không có mô hình đi kèm. **Không huấn luyện lại (`.fit()`), không tối ưu ngưỡng bằng Test.**

**Quyết định đã cố định TRƯỚC khi mở Test:** `LogisticRegression` (Bước 5) và threshold `0.225` (Bước 6A Validation). Rule thử nghiệm: ngưỡng cao nhất trong lưới đã thử với Recall Validation ≥80%. Mốc 80% là mục tiêu kỹ thuật tùy chọn sau khi xem Validation, **không phải tiêu chuẩn y khoa**; cách chọn ngưỡng mang tính exploratory.

**Chỉ được chạy Test một lần**. Nếu kết quả kém, ghi nhận trung thực; không sử dụng Test để cải thiện model/threshold và gọi đó là test độc lập. Kết quả chỉ đánh giá phân loại dấu hiệu HbA1c ≥5,7% trong cohort đang nghiên cứu; không chẩn đoán, không dự báo mắc type 2 trong tương lai.

Các đầu ra lưu dữ liệu **tổng hợp**; không xuất kết quả cá thể.

## 0. Điều kiện cần có từ các bước trước

Notebook cần các file **trên máy bạn**:
- `models/baseline/nhanes_hba1c_baseline_step5.joblib`
- `data/outputs/model_training/step5_metadata.json`
- `data/outputs/model_training/validation.csv` (chỉ dùng xác minh ID tập tách)
- `data/outputs/model_training/train.csv` (chỉ dùng xác minh ID tập tách)
- `data/outputs/model_training/test_holdout_DO_NOT_TUNE.csv` (chỉ mở ở ô được đánh dấu rõ)
- `data/outputs/model_evaluation/validation_6a/06a_validation_summary.json`
- `data/outputs/model_evaluation/validation_6a/06a_validation_thresholds.csv`
- `06b_locked_evaluation_plan.json` (đi kèm tài liệu này).

**Chạy hai ô kiểm tra trước; chúng hoàn toàn không đọc Test.**

In [2]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
import joblib
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / "data" / "outputs" / "model_training").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "outputs" / "model_training").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Mở Jupyter trong thư mục 03-ML-Research hoặc thư mục notebooks của nó.")

TRAIN_DIR = PROJECT_ROOT / "data" / "outputs" / "model_training"
VAL_DIR = PROJECT_ROOT / "data" / "outputs" / "model_evaluation" / "validation_6a"
OUT_DIR = PROJECT_ROOT / "data" / "outputs" / "model_evaluation" / "test_6b_FINAL"
PLAN_FILE = PROJECT_ROOT / "06b_locked_evaluation_plan.json"
MODEL_FILE = PROJECT_ROOT / "models" / "baseline" / "nhanes_hba1c_baseline_step5.joblib"
META_FILE = TRAIN_DIR / "step5_metadata.json"
VAL_SUMMARY_FILE = VAL_DIR / "06a_validation_summary.json"
VAL_THRESHOLDS = VAL_DIR / "06a_validation_thresholds.csv"
TRAIN_FILE = TRAIN_DIR / "train.csv"
VAL_FILE = TRAIN_DIR / "validation.csv"
TEST_FILE = TRAIN_DIR / "test_holdout_DO_NOT_TUNE.csv"
FINAL_FILE = OUT_DIR / "06b_test_final_results.json"

for p in [PLAN_FILE, MODEL_FILE, META_FILE, VAL_SUMMARY_FILE, VAL_THRESHOLDS, TRAIN_FILE, VAL_FILE, TEST_FILE]:
    if not p.is_file():
        raise FileNotFoundError(f"Thiếu file cần thiết: {p}")
print("Python", platform.python_version(), "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
print("Preflight: chỉ kiểm tra đường dẫn tồn tại; CHƯA ĐỌC TEST.")

Python 3.13.9 | pandas 3.0.0 | sklearn 1.8.0
Preflight: chỉ kiểm tra đường dẫn tồn tại; CHƯA ĐỌC TEST.


In [3]:
# ===== PRE-FLIGHT: KHÔNG MỞ FILE TEST, KHÔNG .FIT() =====
plan = json.loads(PLAN_FILE.read_text(encoding="utf-8"))
meta = json.loads(META_FILE.read_text(encoding="utf-8"))
val_summary = json.loads(VAL_SUMMARY_FILE.read_text(encoding="utf-8"))
val_thresholds = pd.read_csv(VAL_THRESHOLDS)

FEATURES = ["RIDAGEYR", "RIAGENDR", "BMXBMI"]
TARGET = "a1c_ge_57_candidate"
threshold = float(plan["threshold"])
assert plan["selected_model"] == meta["selected_model"] == "LogisticRegression"
assert plan["features"] == meta["features"] == FEATURES
assert plan["target"] == TARGET
assert threshold == 0.225, "Threshold thay đổi sau Bước 6A!"
assert len(val_thresholds) >= 15
v = val_thresholds.loc[np.isclose(val_thresholds["threshold"], threshold)]
assert len(v) == 1
v = v.iloc[0]
for key in ["tp", "fn", "fp", "tn"]:
    assert int(v[key]) == int(plan["validation_reference"][key]), f"Khác Validation: {key}"
assert abs(float(v["recall"]) - float(plan["validation_reference"]["recall"])) < 1e-8
assert val_summary["validation_only"] is True and val_summary["test_opened"] is False
assert val_summary["fit_executed"] is False
assert val_summary["model"] == "LogisticRegression"
assert val_summary["validation_n"] == 705 and val_summary["validation_positive_n"] == 208
assert meta["test_holdout_sample_count"] == 705
assert meta["test_metrics"] is None
assert meta["sklearn_version"] == sklearn.__version__, "Hãy chạy cùng môi trường scikit-learn đã dùng ở Bước 5"
model_sha = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()
assert model_sha == val_summary["model_sha256"], "Model đã đổi sau Validation!"
train_ids = set(pd.read_csv(TRAIN_FILE, usecols=["SEQN"])["SEQN"].astype(str))
val_ids = set(pd.read_csv(VAL_FILE, usecols=["SEQN"])["SEQN"].astype(str))
assert train_ids.isdisjoint(val_ids)
assert not FINAL_FILE.exists(), "Đã có kết quả Test cuối: KHÔNG chạy lại hoặc chọn ngưỡng theo Test."

print("PRE-FLIGHT PASSED — model + threshold đã cố định:", meta["selected_model"], "/",threshold)
print("Model SHA256:",model_sha[:24],"...")
print("Validation reference: TP=173, FN=35, FP=249, TN=248 | Test vẫn chưa được đọc.")

PRE-FLIGHT PASSED — model + threshold đã cố định: LogisticRegression / 0.225
Model SHA256: 31e9c94b0f14369be7920b93 ...
Validation reference: TP=173, FN=35, FP=249, TN=248 | Test vẫn chưa được đọc.


## 1. ⚠️ CHỈ KHI BẠN CHỦ ĐỘNG CHẠY Ô TIẾP THEO, TEST MỚI ĐƯỢC MỞ

Đây là lần đánh giá Test holdout **duy nhất** theo kế hoạch đã khóa. Tất cả mô hình/feature/threshold đã chốt trước khi đọc Test. Sau khi chạy **không** điều chỉnh threshold và đánh giá Test lần thứ hai. Đừng chạy ô nếu muốn thay đổi quyết định Validation; khi đó chúng ta phải xem lại kế hoạch mà vẫn giữ Test đóng.

In [4]:
# ===== FINAL TEST: BẠN CHỦ ĐỘNG CHẠY Ô NÀY =====
# Ô này KHÔNG có .fit() hoặc train lại. Chỉ tính metric Test tại threshold ĐÃ KHÓA.
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score,
    brier_score_loss, precision_score, recall_score, f1_score, fbeta_score,
    accuracy_score, balanced_accuracy_score)
from datetime import datetime, timezone

if FINAL_FILE.exists():
    raise RuntimeError("Kết quả Test đã tồn tại; không lặp lại đánh giá holdout.")

# Chỉ nạp model joblib được CHÍNH BẠN tạo tin cậy ở Bước 5.
model = joblib.load(MODEL_FILE)
assert list(model.classes_) == [0, 1]
test = pd.read_csv(TEST_FILE)  # MỞ TEST LẦN ĐẦU Ở ĐÂY
assert len(test) == meta["test_holdout_sample_count"] == 705
assert {"SEQN", TARGET, *FEATURES}.issubset(test.columns)
assert test["SEQN"].notna().all() and test["SEQN"].is_unique
assert test[TARGET].notna().all() and set(test[TARGET].unique()) <= {0, 1}
test_ids = set(test["SEQN"].astype(str))
assert test_ids.isdisjoint(train_ids) and test_ids.isdisjoint(val_ids), "Trùng SEQN giữa các split"

X_test, y_test = test[FEATURES], test[TARGET].astype(int)
p1 = model.predict_proba(X_test)[:, 1]
assert np.isfinite(p1).all() and ((p1>=0)&(p1<=1)).all()
pred = (p1 >= threshold).astype(int)
tn,fp,fn,tp = map(int, confusion_matrix(y_test, pred, labels=[0,1]).ravel())

out = {
  "experiment":"EXP-EVAL-TEST-001", "phase":"phase3_step6b", "test_evaluated":True,
  "evaluated_at_utc":datetime.now(timezone.utc).isoformat(),
  "plan_id":plan["plan_id"], "model":meta["selected_model"], "model_sha256":model_sha,
  "threshold_locked_before_test":threshold,"features":FEATURES,
  "n_test":int(len(test)), "positive_n":int(y_test.sum()),
  "positive_rate_unweighted":float(y_test.mean()),
  "roc_auc":float(roc_auc_score(y_test,p1)),
  "average_precision":float(average_precision_score(y_test,p1)),
  "brier_score":float(brier_score_loss(y_test,p1)),
  "precision":float(precision_score(y_test,pred,zero_division=0)),
  "recall":float(recall_score(y_test,pred,zero_division=0)),
  "specificity":float(tn/(tn+fp)) if tn+fp else None,
  "f1":float(f1_score(y_test,pred,zero_division=0)),
  "f2":float(fbeta_score(y_test,pred,beta=2,zero_division=0)),
  "accuracy":float(accuracy_score(y_test,pred)),
  "balanced_accuracy":float(balanced_accuracy_score(y_test,pred)),
  "confusion_matrix":{"tn":tn,"fp":fp,"fn":fn,"tp":tp},
  "pred_positive_n":int(pred.sum()),
  "pred_positive_pct":float(pred.mean()*100),
  "survey_weighted":False, "external_validation":False, "clinical_validation":False,
  "fit_executed":False,
  "limitations":["NHANES cross-sectional: only classifies observed HbA1c >=5.7%",
    "not a future type 2 diabetes risk estimate or diagnosis", "survey weights not used",
    "pregnancy status unavailable for many respondents", "test used once at locked threshold"]
}
OUT_DIR.mkdir(parents=True,exist_ok=True)
# Chỉ ghi tổng hợp. Tên file có x để ngăn overwrite nếu cell vô tình chạy lại.
with FINAL_FILE.open("x",encoding="utf-8") as f:
    json.dump(out,f,indent=2,ensure_ascii=False)

print("===== BƯỚC 6B — FINAL TEST HOLDOUT (NGƯỜI DÙNG TỰ CHẠY) =====")
for k in ["model","n_test","positive_n","threshold_locked_before_test","roc_auc",
          "average_precision","brier_score","precision","recall","specificity",
          "f1","f2","accuracy","balanced_accuracy","pred_positive_n","pred_positive_pct"]:
    v=out[k]
    print(f"{k}: {v:.4f}" if isinstance(v,float) else f"{k}: {v}")
print("Confusion Matrix:",out["confusion_matrix"])
print("Saved:",FINAL_FILE)
print("FIT: NO | THRESHOLD TUNING ON TEST: NO | EXTERNAL VALIDATION: NO")
print("===== KẾT THÚC 6B — GỬI OUTPUT CHO TÔI =====")

===== BƯỚC 6B — FINAL TEST HOLDOUT (NGƯỜI DÙNG TỰ CHẠY) =====
model: LogisticRegression
n_test: 705
positive_n: 208
threshold_locked_before_test: 0.2250
roc_auc: 0.7406
average_precision: 0.5118
brier_score: 0.1785
precision: 0.4070
recall: 0.8413
specificity: 0.4869
f1: 0.5486
f2: 0.6933
accuracy: 0.5915
balanced_accuracy: 0.6641
pred_positive_n: 430
pred_positive_pct: 60.9929
Confusion Matrix: {'tn': 242, 'fp': 255, 'fn': 33, 'tp': 175}
Saved: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_evaluation\test_6b_FINAL\06b_test_final_results.json
FIT: NO | THRESHOLD TUNING ON TEST: NO | EXTERNAL VALIDATION: NO
===== KẾT THÚC 6B — GỬI OUTPUT CHO TÔI =====


## 2. Dừng tại đây, gửi kết quả cho tôi

Gửi **toàn bộ output cuối** và (nếu thuận tiện) file `06b_test_final_results.json`. Không gửi `test_holdout_DO_NOT_TUNE.csv`, dữ liệu cá thể hoặc tệp `.joblib`.

**Không chạy lại để có kết quả đẹp hơn.** Nếu kết quả không đạt mong đợi, chúng ta ghi nhận giới hạn và lên kế hoạch một nghiên cứu/mẫu kiểm thử mới riêng biệt. Không khẳng định mô hình đủ dùng lâm sàng chỉ từ một phép đánh giá nội bộ.